In [ ]:
!pip install -q scikit-learn joblib pyarrow

In [ ]:
import os
import re
import gc
import json
import math
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 240)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
LOCAL_ARTIFACTS_DIR = Path("artifacts")
DRIVE_ARTIFACTS_DIR = Path("/content/drive/MyDrive/tablewise/artifacts_new")

if DRIVE_ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = DRIVE_ARTIFACTS_DIR
elif LOCAL_ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = LOCAL_ARTIFACTS_DIR
else:
    raise FileNotFoundError("Nu am găsit artifacts/. Rulează notebookurile anterioare.")

FAISS_DIR = ARTIFACTS_DIR / "faiss"
MAPPING_PATH = FAISS_DIR / "restaurant_index_mapping.parquet"

OUTPUT_DIR = ARTIFACTS_DIR / "feedback_rlhf"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = OUTPUT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = OUTPUT_DIR / "reward_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

METRICS_DIR = OUTPUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("MAPPING_PATH:", MAPPING_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

ARTIFACTS_DIR: /content/drive/MyDrive/tablewise/artifacts_new
MAPPING_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet
OUTPUT_DIR: /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf


In [ ]:
if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"Missing mapping file: {MAPPING_PATH}")

restaurants_df = pd.read_parquet(MAPPING_PATH)

print("restaurants_df shape:", restaurants_df.shape)
print("Columns:")
print(restaurants_df.columns.tolist())

display(restaurants_df.head(3))

restaurants_df shape: (1033798, 31)
Columns:
['faiss_idx', 'restaurant_id', 'name', 'country', 'region', 'province', 'city', 'city_filled', 'city_filled_norm', 'city_source', 'address', 'latitude', 'longitude', 'rating', 'price_level', 'price_bucket', 'top_tags_text', 'top_tags_norm_list', 'meals_text', 'meals_norm_list', 'features_text', 'features_norm_list', 'special_diets_text', 'special_diets_norm_list', 'popularity_detailed', 'popularity_rank_num', 'popularity_total_num', 'popularity_score', 'profile_quality_score', 'short_profile', 'profile_text']


,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text
0,0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Features: Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Servi..."
1,1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaurants in Saint-Jouvent."
2,2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch, Drinks. Features: Reservations, Seating, Table Service, Wheelchair Accessible. Popularity: #1 of 1..."


In [ ]:
def normalize_text(x):
    if x is None or pd.isna(x):
        return ""

    s = str(x).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def parse_possible_list(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, tuple):
        return list(x)

    s = str(x).strip()

    if not s or s.lower() in {"nan", "none", "null", "unknown"}:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [str(v) for v in parsed]
        except Exception:
            pass

    return [v.strip() for v in re.split(r"[,|;/]", s) if v.strip()]


def list_norm(xs):
    return [normalize_text(x) for x in parse_possible_list(xs) if normalize_text(x)]


def list_contains_term(row_list, term):
    term_norm = normalize_text(term)

    if not term_norm:
        return False

    if not isinstance(row_list, list):
        return False

    for item in row_list:
        item_norm = normalize_text(item)

        if not item_norm:
            continue

        if item_norm == term_norm:
            return True

        if term_norm in item_norm:
            return True

        if item_norm in term_norm and len(item_norm) >= 4:
            return True

    return False

In [ ]:
df_reward_base = restaurants_df.copy()

required_base_cols = [
    "restaurant_id",
    "name",
    "city_filled",
    "country",
    "price_bucket",
    "rating",
    "popularity_score",
]

missing_base = [c for c in required_base_cols if c not in df_reward_base.columns]
assert not missing_base, f"Lipsesc coloane necesare: {missing_base}"

df_reward_base["city_filled_norm"] = df_reward_base["city_filled"].apply(normalize_text)
df_reward_base["country_norm"] = df_reward_base["country"].apply(normalize_text)
df_reward_base["price_bucket"] = df_reward_base["price_bucket"].apply(
    lambda x: normalize_text(x) if pd.notna(x) else None
)

for col in ["top_tags", "meals", "features", "special_diets"]:
    text_col = f"{col}_text"
    list_col = f"{col}_list"
    norm_col = f"{col}_norm_list"

    if norm_col in df_reward_base.columns:
        df_reward_base[norm_col] = df_reward_base[norm_col].apply(list_norm)
    elif list_col in df_reward_base.columns:
        df_reward_base[norm_col] = df_reward_base[list_col].apply(list_norm)
    elif text_col in df_reward_base.columns:
        df_reward_base[norm_col] = df_reward_base[text_col].apply(list_norm)
    else:
        df_reward_base[norm_col] = [[] for _ in range(len(df_reward_base))]

df_reward_base["rating"] = pd.to_numeric(df_reward_base["rating"], errors="coerce")
df_reward_base["rating_score"] = (df_reward_base["rating"] / 5.0).clip(0, 1).fillna(0.5)

df_reward_base["popularity_score"] = pd.to_numeric(
    df_reward_base["popularity_score"],
    errors="coerce",
).clip(0, 1).fillna(0.5)

df_reward_base = df_reward_base[df_reward_base["city_filled_norm"].astype(str).str.len() > 0].copy()
df_reward_base = df_reward_base.reset_index(drop=True)

print("df_reward_base shape:", df_reward_base.shape)

display(df_reward_base[
    [
        "restaurant_id",
        "name",
        "city_filled",
        "country",
        "price_bucket",
        "rating",
        "rating_score",
        "popularity_score",
        "top_tags_norm_list",
        "special_diets_norm_list",
        "meals_norm_list",
        "features_norm_list",
    ]
].head())

df_reward_base shape: (1033798, 33)


,restaurant_id,name,city_filled,country,price_bucket,rating,rating_score,popularity_score,top_tags_norm_list,special_diets_norm_list,meals_norm_list,features_norm_list
0,g10001637-d10002227,Le 147,Saint-Jouvent,France,cheap,4.0,0.8,0.36907,[cheap eats french],[],[lunch dinner],[reservations seating wheelchair accessible serves alcohol accepts credit cards table service]
1,g10001637-d14975787,Le Saint Jouvent,Saint-Jouvent,France,cheap,4.0,0.8,0.00000,[cheap eats],[],[],[]
2,g10002858-d4586832,Au Bout du Pont,Rivarennes,France,cheap,5.0,1.0,0.50000,[cheap eats french european],[],[dinner lunch drinks],[reservations seating table service wheelchair accessible]
3,g10002986-d3510044,Le Relais de Naiade,Lacelle,France,cheap,4.0,0.8,0.50000,[cheap eats french],[],[lunch dinner],[reservations seating serves alcohol table service wheelchair accessible]
4,g10022428-d9767191,Relais Du MontSeigne,Saint-Laurent-de-Levezou,France,mid,4.5,0.9,0.50000,[mid-range french],[],[lunch dinner],[reservations seating wheelchair accessible table service]


In [ ]:
CITIES_FOR_FEEDBACK = [
    "rome",
    "barcelona",
    "paris",
    "madrid",
    "milan",
    "lisbon",
    "athens",
    "porto",
    "florence",
    "venice",
    "naples",
    "valencia",
    "seville",
    "granada",
    "nice",
    "lyon",
    "marseille",
    "bordeaux",
    "vienna",
    "prague",
    "amsterdam",
    "brussels",
    "budapest",
]

existing_cities = set(df_reward_base["city_filled_norm"].dropna().unique())

CITIES_FOR_FEEDBACK = [c for c in CITIES_FOR_FEEDBACK if c in existing_cities]

print("Cities used for feedback:", CITIES_FOR_FEEDBACK)
print("Num cities:", len(CITIES_FOR_FEEDBACK))

PRICES_FOR_FEEDBACK = ["cheap", "mid", "expensive"]

TAGS_FOR_FEEDBACK = [
    "italian",
    "french",
    "spanish",
    "tapas",
    "greek",
    "seafood",
    "mediterranean",
    "coffee",
    "bar",
    "fast food",
    "fine dining",
]

DIETARY_FOR_FEEDBACK = [
    "vegetarian friendly",
    "vegan options",
    "gluten free options",
]

MEALS_FOR_FEEDBACK = [
    "breakfast",
    "brunch",
    "lunch",
    "dinner",
    "drinks",
]

FEATURES_FOR_FEEDBACK = [
    "outdoor seating",
    "reservations",
    "family friendly",
    "romantic",
]

Cities used for feedback: ['rome', 'barcelona', 'paris', 'madrid', 'milan', 'lisbon', 'athens', 'porto', 'florence', 'venice', 'naples', 'valencia', 'seville', 'granada', 'nice', 'lyon', 'marseille', 'bordeaux', 'vienna', 'prague', 'amsterdam', 'budapest']
Num cities: 22


In [ ]:
QUERY_PATTERNS_FEEDBACK = [
    "{price} restaurant in {city}",
    "{tag} restaurant in {city}",
    "{price} {tag} restaurant in {city}",
    "{dietary} restaurant in {city}",
    "{price} {dietary} restaurant in {city}",
    "{dietary} {tag} restaurant in {city}",
    "{price} {dietary} {tag} restaurant in {city}",
    "{tag} place for {meal} in {city}",
    "{price} place for {meal} in {city}",
    "{feature} restaurant in {city}",
    "{price} restaurant with {feature} in {city}",
    "{dietary} place for {meal} in {city}",
    "{price} {tag} place with {feature} in {city}",
    "can you recommend a {price} restaurant in {city}",
    "i want a {tag} place in {city}",
    "find me a {dietary} restaurant in {city}",
    "looking for {tag} food for {meal} in {city}",
    "i need a {price} place in {city} with {feature}",
    "somewhere {feature} for {meal} in {city}",
    "any good {price} {tag} spots in {city}",
    "can you suggest a {dietary} {tag} place in {city}",
]


def sample_constraint_query():
    city = random.choice(CITIES_FOR_FEEDBACK)

    parsed = {
        "city": city,
        "country": None,
        "price_bucket": None,
        "tags": [],
        "dietary": [],
        "matched_meals": [],
        "matched_features": [],
    }

    pattern = random.choice(QUERY_PATTERNS_FEEDBACK)

    values = {
        "city": city.title(),
    }

    if "{price}" in pattern:
        price = random.choice(PRICES_FOR_FEEDBACK)
        parsed["price_bucket"] = price
        values["price"] = price

    if "{tag}" in pattern:
        tag = random.choice(TAGS_FOR_FEEDBACK)
        parsed["tags"] = [tag]
        values["tag"] = tag

    if "{dietary}" in pattern:
        dietary = random.choice(DIETARY_FOR_FEEDBACK)
        parsed["dietary"] = [dietary]
        values["dietary"] = dietary

    if "{meal}" in pattern:
        meal = random.choice(MEALS_FOR_FEEDBACK)
        parsed["matched_meals"] = [meal]
        values["meal"] = meal

    if "{feature}" in pattern:
        feature = random.choice(FEATURES_FOR_FEEDBACK)
        parsed["matched_features"] = [feature]
        values["feature"] = feature

    query = pattern.format(**values)

    return query, parsed


for _ in range(5):
    q, p = sample_constraint_query()
    print(q)
    print(json.dumps(p, indent=2))
    print()

vegetarian friendly restaurant in Amsterdam
{
  "city": "amsterdam",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

tapas place for brunch in Florence
{
  "city": "florence",
  "country": null,
  "price_bucket": null,
  "tags": [
    "tapas"
  ],
  "dietary": [],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}

i need a cheap place in Madrid with romantic
{
  "city": "madrid",
  "country": null,
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": [
    "romantic"
  ]
}

cheap restaurant in Barcelona
{
  "city": "barcelona",
  "country": null,
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

bar place for drinks in Athens
{
  "city": "athens",
  "country": null,
  "price_bucket": null,
  "tags": [
    "bar"
  ],
  "dietary": [],
  "matched_meals": [
    "

In [ ]:
def compute_candidate_scores(row, parsed):
    city = parsed.get("city")
    country = parsed.get("country")
    price = parsed.get("price_bucket")
    tags = parsed.get("tags", [])
    dietary = parsed.get("dietary", [])
    meals = parsed.get("matched_meals", [])
    features = parsed.get("matched_features", [])

    hard_constraint_score = 1.0

    if city and row.get("city_filled_norm") != city:
        hard_constraint_score = 0.0

    if country and row.get("country_norm") != country:
        hard_constraint_score = 0.0

    max_soft = 0
    soft_hits = 0
    metadata_points = 0
    max_metadata_points = 0

    if price:
        max_soft += 1
        max_metadata_points += 2

        if row.get("price_bucket") == price:
            soft_hits += 1
            metadata_points += 2

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        max_soft += 1
        max_metadata_points += 2

        if list_contains_term(row_tags, tag):
            soft_hits += 1
            metadata_points += 2

    for diet in dietary:
        max_soft += 1
        max_metadata_points += 2

        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            soft_hits += 1
            metadata_points += 2

    for meal in meals:
        max_soft += 1
        max_metadata_points += 1

        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            soft_hits += 1
            metadata_points += 1

    for feature in features:
        max_soft += 1
        max_metadata_points += 1

        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            soft_hits += 1
            metadata_points += 1

    if max_soft > 0:
        soft_constraint_score = soft_hits / max_soft
    else:
        soft_constraint_score = 0.5

    if max_metadata_points > 0:
        metadata_score_norm = metadata_points / max_metadata_points
    else:
        metadata_score_norm = 0.0

    metadata_score_norm = float(np.clip(metadata_score_norm, 0, 1))

    rating_score = float(row.get("rating_score", 0.5) or 0.5)
    popularity_score_norm = float(row.get("popularity_score", 0.5) or 0.5)

    semantic_score_norm = (
        0.45 * soft_constraint_score
        + 0.25 * metadata_score_norm
        + 0.15 * rating_score
        + 0.15 * np.random.uniform(0.2, 1.0)
    )

    semantic_score_norm = float(np.clip(semantic_score_norm, 0, 1))

    final_rerank_score = (
        0.40 * semantic_score_norm
        + 0.20 * metadata_score_norm
        + 0.15 * rating_score
        + 0.10 * popularity_score_norm
        + 0.15 * soft_constraint_score
    )

    final_rerank_score *= hard_constraint_score
    final_rerank_score = float(np.clip(final_rerank_score, 0, 1))

    return {
        "semantic_score_norm": semantic_score_norm,
        "metadata_score_norm": metadata_score_norm,
        "rating_score": rating_score,
        "popularity_score_norm": popularity_score_norm,
        "soft_constraint_score": float(soft_constraint_score),
        "hard_constraint_score": float(hard_constraint_score),
        "final_rerank_score": final_rerank_score,
    }

In [ ]:
def safe_sample(df_in, n):
    if len(df_in) == 0:
        return df_in

    return df_in.sample(
        min(n, len(df_in)),
        random_state=random.randint(1, 10_000_000),
    )


city_groups = {
    city: group
    for city, group in df_reward_base.groupby("city_filled_norm", sort=False)
}

all_indices = df_reward_base.index.to_numpy()


def sample_candidates_for_query_fast(
    parsed,
    n_positive=8,
    n_partial=8,
    n_negative=8,
    city_pool_max=3000,
    global_negative_pool=5000,
):
    city = parsed.get("city")
    price = parsed.get("price_bucket")
    tags = parsed.get("tags", [])
    dietary = parsed.get("dietary", [])
    meals = parsed.get("matched_meals", [])
    features = parsed.get("matched_features", [])

    if city and city in city_groups:
        city_df = city_groups[city]
    else:
        city_df = df_reward_base

    city_pool = safe_sample(city_df, city_pool_max)

    pos_mask = pd.Series(True, index=city_pool.index)

    if price:
        pos_mask &= city_pool["price_bucket"].eq(price)

    for tag in tags:
        pos_mask &= city_pool["top_tags_norm_list"].apply(lambda xs: list_contains_term(xs, tag))

    for diet in dietary:
        pos_mask &= (
            city_pool["special_diets_norm_list"].apply(lambda xs: list_contains_term(xs, diet))
            | city_pool["top_tags_norm_list"].apply(lambda xs: list_contains_term(xs, diet))
            | city_pool["features_norm_list"].apply(lambda xs: list_contains_term(xs, diet))
        )

    for meal in meals:
        pos_mask &= (
            city_pool["meals_norm_list"].apply(lambda xs: list_contains_term(xs, meal))
            | city_pool["top_tags_norm_list"].apply(lambda xs: list_contains_term(xs, meal))
        )

    for feature in features:
        pos_mask &= (
            city_pool["features_norm_list"].apply(lambda xs: list_contains_term(xs, feature))
            | city_pool["top_tags_norm_list"].apply(lambda xs: list_contains_term(xs, feature))
        )

    positives = city_pool[pos_mask].copy()

    partials = city_pool.copy()

    if city:
        other_city_pool = df_reward_base[df_reward_base["city_filled_norm"] != city]
    else:
        other_city_pool = df_reward_base

    negative_pool = safe_sample(other_city_pool, global_negative_pool)

    pos_sample = safe_sample(positives, n_positive)
    partial_sample = safe_sample(partials, n_partial)
    neg_sample = safe_sample(negative_pool, n_negative)

    candidates = pd.concat(
        [pos_sample, partial_sample, neg_sample],
        ignore_index=True,
    )

    candidates = candidates.drop_duplicates(subset=["restaurant_id"]).reset_index(drop=True)

    return candidates

In [ ]:
def hidden_user_utility(row_scores):

    utility = (
        0.30 * row_scores["hard_constraint_score"]
        + 0.25 * row_scores["soft_constraint_score"]
        + 0.15 * row_scores["metadata_score_norm"]
        + 0.20 * row_scores["rating_score"]
        + 0.10 * row_scores["popularity_score_norm"]
    )

    utility += np.random.normal(0, 0.10)

    return float(np.clip(utility, 0, 1))


def utility_to_relevance(utility):
    if utility >= 0.78:
        return 2
    if utility >= 0.55:
        return 1
    return 0


def relevance_to_feedback(rel):
    if rel == 2:
        return 1
    if rel == 1:
        return 0
    return -1

In [ ]:
import time

N_SYNTHETIC_QUERIES = 300
N_POSITIVE_PER_QUERY = 8
N_PARTIAL_PER_QUERY = 8
N_NEGATIVE_PER_QUERY = 8

feedback_rows = []

start_time = time.time()

for query_idx in range(N_SYNTHETIC_QUERIES):
    if query_idx > 0 and query_idx % 50 == 0:
        elapsed = time.time() - start_time
        print(f"Generated {query_idx}/{N_SYNTHETIC_QUERIES} queries in {elapsed/60:.2f} min")

    query, parsed = sample_constraint_query()

    candidates = sample_candidates_for_query_fast(
        parsed,
        n_positive=N_POSITIVE_PER_QUERY,
        n_partial=N_PARTIAL_PER_QUERY,
        n_negative=N_NEGATIVE_PER_QUERY,
        city_pool_max=3000,
        global_negative_pool=5000,
    )

    for _, row in candidates.iterrows():
        scores = compute_candidate_scores(row, parsed)
        utility = hidden_user_utility(scores)
        relevance = utility_to_relevance(utility)
        feedback = relevance_to_feedback(relevance)

        feedback_rows.append({
            "query_id": query_idx,
            "query": query,
            "parsed_query_json": json.dumps(parsed, ensure_ascii=False),
            "restaurant_id": str(row.get("restaurant_id")),
            "name": row.get("name"),
            "city_filled": row.get("city_filled"),
            "country": row.get("country"),
            "price_bucket": row.get("price_bucket"),
            "rating": row.get("rating"),
            "feedback": feedback,
            "relevance": relevance,
            "hidden_utility": utility,
            **scores,
        })

feedback_labeled_df = pd.DataFrame(feedback_rows)

feedback_source = "synthetic_large_non_circular_fast"

elapsed = time.time() - start_time

print("Feedback source:", feedback_source)
print("Rows:", len(feedback_labeled_df))
print("Queries:", feedback_labeled_df["query_id"].nunique())
print(f"Elapsed: {elapsed/60:.2f} min")

print("\nFeedback distribution:")
display(feedback_labeled_df["feedback"].value_counts(dropna=False).sort_index())

print("\nRelevance distribution:")
display(feedback_labeled_df["relevance"].value_counts(dropna=False).sort_index())

display(feedback_labeled_df.head(10))

Generated 50/300 queries in 0.70 min
Generated 100/300 queries in 1.37 min
Generated 150/300 queries in 2.02 min
Generated 200/300 queries in 2.68 min
Generated 250/300 queries in 3.34 min
Feedback source: synthetic_large_non_circular_fast
Rows: 6488
Queries: 300
Elapsed: 4.00 min

Feedback distribution:


,count
feedback,
-1,3626
0,1194
1,1668



Relevance distribution:


,count
relevance,
0,3626
1,1194
2,1668


,query_id,query,parsed_query_json,restaurant_id,name,city_filled,country,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score
0,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d10446547,Bien Etre et Petit Plat,Marseille,France,expensive,4.5,1,2,0.894502,0.977362,1.0,0.9,0.189023,1.0,1.0,0.894847
1,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d1750996,La Broceliande,Marseille,France,mid,4.0,1,2,0.848353,0.852595,1.0,0.8,0.224041,1.0,1.0,0.833442
2,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d4554756,Cafe Borely,Marseille,France,mid,3.5,0,1,0.651879,0.893031,1.0,0.7,0.149436,1.0,1.0,0.827156
3,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d11866683,waoreng Bali,Marseille,France,mid,4.0,1,2,0.863221,0.865859,1.0,0.8,0.020287,1.0,1.0,0.818372
4,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d12438918,Green Love,Marseille,France,expensive,4.5,1,2,1.000000,0.910167,1.0,0.9,0.389458,1.0,1.0,0.888012
5,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d9881313,Bistrot o'prado,Marseille,France,mid,5.0,1,2,1.000000,0.943690,1.0,1.0,0.664194,1.0,1.0,0.943895
6,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d8756125,Noodles Chôp,Marseille,France,mid,4.5,1,2,0.896314,0.935780,1.0,0.9,0.806574,1.0,1.0,0.939969
7,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d1323493,Cafe Bovo,Marseille,France,expensive,5.0,1,2,0.883945,0.899427,1.0,1.0,0.745240,1.0,1.0,0.934295
8,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d6924182,Le Commodore,Marseille,France,mid,3.5,-1,0,0.461590,0.164915,0.0,0.7,0.107117,0.0,1.0,0.181678
9,0,gluten free options restaurant in Marseille,"{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",g187253-d3569100,L'Horloge du Cap Est,Marseille,France,expensive,4.5,-1,0,0.343558,0.232523,0.0,0.9,0.174377,0.0,1.0,0.245447


In [ ]:
print("Rows per query:")
display(feedback_labeled_df.groupby("query_id").size().describe())

print("\nSample by feedback label:")
for fb in [-1, 0, 1]:
    print("=" * 120)
    print("Feedback:", fb)
    display(
        feedback_labeled_df[feedback_labeled_df["feedback"] == fb][
            [
                "query",
                "name",
                "city_filled",
                "price_bucket",
                "rating",
                "feedback",
                "relevance",
                "hidden_utility",
                "semantic_score_norm",
                "metadata_score_norm",
                "rating_score",
                "popularity_score_norm",
                "soft_constraint_score",
                "hard_constraint_score",
                "final_rerank_score",
            ]
        ].head(5)
    )

Rows per query:


,0
count,300.000000
mean,21.626667
std,3.535859
min,16.000000
25%,16.000000
50%,24.000000
75%,24.000000
max,24.000000



Sample by feedback label:
Feedback: -1


,query,name,city_filled,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score
8,gluten free options restaurant in Marseille,Le Commodore,Marseille,mid,3.5,-1,0,0.461590,0.164915,0.0,0.7,0.107117,0.0,1.0,0.181678
9,gluten free options restaurant in Marseille,L'Horloge du Cap Est,Marseille,expensive,4.5,-1,0,0.343558,0.232523,0.0,0.9,0.174377,0.0,1.0,0.245447
10,gluten free options restaurant in Marseille,Ashoka,Marseille,mid,4.0,-1,0,0.538185,0.263394,0.0,0.8,0.249386,0.0,1.0,0.250296
11,gluten free options restaurant in Marseille,La Boite a Panisse,Marseille,cheap,4.5,-1,0,0.396024,0.263071,0.0,0.9,0.304390,0.0,1.0,0.270667
12,gluten free options restaurant in Marseille,Burger Dream,Marseille,unknown,NaN,-1,0,0.363949,0.172229,0.0,0.5,0.500000,0.0,1.0,0.193892


Feedback: 0


,query,name,city_filled,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score
2,gluten free options restaurant in Marseille,Cafe Borely,Marseille,mid,3.5,0,1,0.651879,0.893031,1.0,0.7,0.149436,1.0,1.0,0.827156
25,vegan options restaurant in Vienna,Formosa,Vienna,mid,4.0,0,1,0.718880,0.950268,1.0,0.8,0.099850,1.0,1.0,0.860092
26,vegan options restaurant in Vienna,Conditorei Sluka Kärntner Strasse,Vienna,expensive,4.0,0,1,0.698556,0.966066,1.0,0.8,0.438435,1.0,1.0,0.900270
27,vegan options restaurant in Vienna,Hamama,Vienna,mid,3.5,0,1,0.731283,0.936586,1.0,0.7,0.093668,1.0,1.0,0.839001
33,vegan options restaurant in Vienna,Donuteria,Vienna,cheap,4.5,0,1,0.601438,0.185713,0.0,0.9,0.097513,0.0,1.0,0.219036


Feedback: 1


,query,name,city_filled,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score
0,gluten free options restaurant in Marseille,Bien Etre et Petit Plat,Marseille,expensive,4.5,1,2,0.894502,0.977362,1.0,0.9,0.189023,1.0,1.0,0.894847
1,gluten free options restaurant in Marseille,La Broceliande,Marseille,mid,4.0,1,2,0.848353,0.852595,1.0,0.8,0.224041,1.0,1.0,0.833442
3,gluten free options restaurant in Marseille,waoreng Bali,Marseille,mid,4.0,1,2,0.863221,0.865859,1.0,0.8,0.020287,1.0,1.0,0.818372
4,gluten free options restaurant in Marseille,Green Love,Marseille,expensive,4.5,1,2,1.000000,0.910167,1.0,0.9,0.389458,1.0,1.0,0.888012
5,gluten free options restaurant in Marseille,Bistrot o'prado,Marseille,mid,5.0,1,2,1.000000,0.943690,1.0,1.0,0.664194,1.0,1.0,0.943895


In [ ]:
FEATURE_COLS = [
    "semantic_score_norm",
    "metadata_score_norm",
    "rating_score",
    "popularity_score_norm",
    "soft_constraint_score",
    "hard_constraint_score",
]

for col in FEATURE_COLS:
    feedback_labeled_df[col] = pd.to_numeric(
        feedback_labeled_df[col],
        errors="coerce",
    ).fillna(0.0)

feedback_labeled_df["final_rerank_score"] = pd.to_numeric(
    feedback_labeled_df["final_rerank_score"],
    errors="coerce",
).fillna(0.0)

feedback_labeled_df["reward_label"] = (feedback_labeled_df["feedback"] == 1).astype(int)

print("Feature columns:", FEATURE_COLS)

print("\nReward label distribution:")
display(feedback_labeled_df["reward_label"].value_counts(dropna=False))

display(feedback_labeled_df[
    ["query", "name", "feedback", "relevance", "reward_label"] + FEATURE_COLS + ["final_rerank_score"]
].head(20))

Feature columns: ['semantic_score_norm', 'metadata_score_norm', 'rating_score', 'popularity_score_norm', 'soft_constraint_score', 'hard_constraint_score']

Reward label distribution:


,count
reward_label,
0,4820
1,1668


,query,name,feedback,relevance,reward_label,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score
0,gluten free options restaurant in Marseille,Bien Etre et Petit Plat,1,2,1,0.977362,1.0,0.9,0.189023,1.0,1.0,0.894847
1,gluten free options restaurant in Marseille,La Broceliande,1,2,1,0.852595,1.0,0.8,0.224041,1.0,1.0,0.833442
2,gluten free options restaurant in Marseille,Cafe Borely,0,1,0,0.893031,1.0,0.7,0.149436,1.0,1.0,0.827156
3,gluten free options restaurant in Marseille,waoreng Bali,1,2,1,0.865859,1.0,0.8,0.020287,1.0,1.0,0.818372
4,gluten free options restaurant in Marseille,Green Love,1,2,1,0.910167,1.0,0.9,0.389458,1.0,1.0,0.888012
5,gluten free options restaurant in Marseille,Bistrot o'prado,1,2,1,0.943690,1.0,1.0,0.664194,1.0,1.0,0.943895
6,gluten free options restaurant in Marseille,Noodles Chôp,1,2,1,0.935780,1.0,0.9,0.806574,1.0,1.0,0.939969
7,gluten free options restaurant in Marseille,Cafe Bovo,1,2,1,0.899427,1.0,1.0,0.745240,1.0,1.0,0.934295
8,gluten free options restaurant in Marseille,Le Commodore,-1,0,0,0.164915,0.0,0.7,0.107117,0.0,1.0,0.181678
9,gluten free options restaurant in Marseille,L'Horloge du Cap Est,-1,0,0,0.232523,0.0,0.9,0.174377,0.0,1.0,0.245447


In [ ]:
groups = feedback_labeled_df["query_id"].values

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        feedback_labeled_df,
        feedback_labeled_df["reward_label"],
        groups,
    )
)

reward_train_df = feedback_labeled_df.iloc[train_idx].copy()
reward_test_df = feedback_labeled_df.iloc[test_idx].copy()

print("Train rows:", len(reward_train_df))
print("Test rows:", len(reward_test_df))
print("Train queries:", reward_train_df["query_id"].nunique())
print("Test queries:", reward_test_df["query_id"].nunique())

print("\nTrain label distribution:")
display(reward_train_df["reward_label"].value_counts(dropna=False))

print("\nTest label distribution:")
display(reward_test_df["reward_label"].value_counts(dropna=False))

Train rows: 4586
Test rows: 1902
Train queries: 210
Test queries: 90

Train label distribution:


,count
reward_label,
0,3374
1,1212



Test label distribution:


,count
reward_label,
0,1446
1,456


In [ ]:
X_train = reward_train_df[FEATURE_COLS].values
y_train = reward_train_df["reward_label"].values

X_test = reward_test_df[FEATURE_COLS].values
y_test = reward_test_df["reward_label"].values

reward_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        max_iter=1000,
    )),
])

reward_model.fit(X_train, y_train)

test_proba = reward_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, test_pred)

try:
    auc = roc_auc_score(y_test, test_proba)
except Exception:
    auc = None

print("Reward model accuracy:", acc)
print("Reward model ROC-AUC:", auc)

print("\nClassification report:")
print(classification_report(y_test, test_pred))

Reward model accuracy: 0.937434279705573
Reward model ROC-AUC: 0.972948666618136

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.93      0.96      1446
           1       0.82      0.95      0.88       456

    accuracy                           0.94      1902
   macro avg       0.90      0.94      0.92      1902
weighted avg       0.94      0.94      0.94      1902



In [ ]:
clf = reward_model.named_steps["clf"]

coef_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "coefficient": clf.coef_[0],
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

display(coef_df)

,feature,coefficient,abs_coefficient
0,hard_constraint_score,2.505017,2.505017
1,metadata_score_norm,1.558147,1.558147
2,soft_constraint_score,1.363923,1.363923
3,rating_score,0.624376,0.624376
4,semantic_score_norm,0.310282,0.310282
5,popularity_score_norm,0.301670,0.301670


In [ ]:
feedback_labeled_df["reward_score"] = reward_model.predict_proba(
    feedback_labeled_df[FEATURE_COLS].values
)[:, 1]

FEEDBACK_RERANK_WEIGHTS = {
    "original_rerank": 0.70,
    "reward_score": 0.30,
}

feedback_labeled_df["feedback_rerank_score"] = (
    FEEDBACK_RERANK_WEIGHTS["original_rerank"] * feedback_labeled_df["final_rerank_score"].astype(float)
    + FEEDBACK_RERANK_WEIGHTS["reward_score"] * feedback_labeled_df["reward_score"].astype(float)
)

display(feedback_labeled_df[
    [
        "query",
        "name",
        "feedback",
        "relevance",
        "final_rerank_score",
        "reward_score",
        "feedback_rerank_score",
    ]
].head(20))

,query,name,feedback,relevance,final_rerank_score,reward_score,feedback_rerank_score
0,gluten free options restaurant in Marseille,Bien Etre et Petit Plat,1,2,0.894847,0.960283,0.914478
1,gluten free options restaurant in Marseille,La Broceliande,1,2,0.833442,0.938094,0.864838
2,gluten free options restaurant in Marseille,Cafe Borely,0,1,0.827156,0.903012,0.849913
3,gluten free options restaurant in Marseille,waoreng Bali,1,2,0.818372,0.916521,0.847817
4,gluten free options restaurant in Marseille,Green Love,1,2,0.888012,0.969226,0.912377
5,gluten free options restaurant in Marseille,Bistrot o'prado,1,2,0.943895,0.987105,0.956858
6,gluten free options restaurant in Marseille,Noodles Chôp,1,2,0.939969,0.984644,0.953372
7,gluten free options restaurant in Marseille,Cafe Bovo,1,2,0.934295,0.988205,0.950468
8,gluten free options restaurant in Marseille,Le Commodore,-1,0,0.181678,0.006411,0.129098
9,gluten free options restaurant in Marseille,L'Horloge du Cap Est,-1,0,0.245447,0.016961,0.176901


In [ ]:
def dcg_at_k(relevances, k):
    rel = np.asarray(relevances[:k], dtype=float)

    if len(rel) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(rel) + 2))
    gains = (2 ** rel - 1)

    return float(np.sum(gains / discounts))


def ndcg_at_k(relevances, k):
    dcg = dcg_at_k(relevances, k)
    ideal = sorted(relevances, reverse=True)
    idcg = dcg_at_k(ideal, k)

    if idcg == 0:
        return None

    return dcg / idcg


def precision_at_k(relevances, k, threshold=1):
    rel = np.asarray(relevances[:k], dtype=float)

    if len(rel) == 0:
        return None

    return float(np.mean(rel >= threshold))


def mrr(relevances, threshold=1):
    for idx, rel in enumerate(relevances, start=1):
        if rel >= threshold:
            return 1.0 / idx

    return 0.0


def evaluate_ranking(df_in, score_col, k_values=[3, 5, 10]):
    rows = []

    for query_id, group in df_in.groupby("query_id"):
        ranked = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        relevances = ranked["relevance"].tolist()

        row = {
            "query_id": query_id,
            "query": ranked["query"].iloc[0],
            "score_col": score_col,
            "num_candidates": len(ranked),
            "mrr": mrr(relevances),
        }

        for k in k_values:
            row[f"precision_at_{k}"] = precision_at_k(relevances, k)
            row[f"ndcg_at_{k}"] = ndcg_at_k(relevances, k)

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
baseline_metrics_by_query = evaluate_ranking(
    feedback_labeled_df,
    score_col="final_rerank_score",
)

feedback_metrics_by_query = evaluate_ranking(
    feedback_labeled_df,
    score_col="feedback_rerank_score",
)

display(baseline_metrics_by_query.head())
display(feedback_metrics_by_query.head())

,query_id,query,score_col,num_candidates,mrr,precision_at_3,ndcg_at_3,precision_at_5,ndcg_at_5,precision_at_10,ndcg_at_10
0,0,gluten free options restaurant in Marseille,final_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,0.8,0.996818
1,1,vegan options restaurant in Vienna,final_rerank_score,24,1.0,1.000000,1.000000,1.0,0.815151,1.0,0.954622
2,2,can you suggest a vegetarian friendly italian place in Nice,final_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,1.0,0.949325
3,3,cheap seafood restaurant in Nice,final_rerank_score,16,0.5,0.666667,0.693426,0.4,0.693426,0.2,0.693426
4,4,can you suggest a gluten free options fast food place in Prague,final_rerank_score,21,1.0,1.000000,1.000000,1.0,1.000000,0.7,0.966160


,query_id,query,score_col,num_candidates,mrr,precision_at_3,ndcg_at_3,precision_at_5,ndcg_at_5,precision_at_10,ndcg_at_10
0,0,gluten free options restaurant in Marseille,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,0.8,0.996818
1,1,vegan options restaurant in Vienna,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,0.815151,1.0,0.958492
2,2,can you suggest a vegetarian friendly italian place in Nice,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,1.0,0.949325
3,3,cheap seafood restaurant in Nice,feedback_rerank_score,16,0.5,0.666667,0.693426,0.4,0.693426,0.2,0.693426
4,4,can you suggest a gluten free options fast food place in Prague,feedback_rerank_score,21,1.0,1.000000,1.000000,1.0,1.000000,0.7,0.966160


In [ ]:
def aggregate_ranking_metrics(metrics_df, method_name):
    numeric_cols = [
        c for c in metrics_df.columns
        if c not in {"query_id", "query", "score_col"}
        and pd.api.types.is_numeric_dtype(metrics_df[c])
    ]

    row = {
        "method": method_name,
    }

    for col in numeric_cols:
        row[col] = metrics_df[col].dropna().mean()

    return row


baseline_agg = aggregate_ranking_metrics(
    baseline_metrics_by_query,
    "baseline_rerank",
)

feedback_agg = aggregate_ranking_metrics(
    feedback_metrics_by_query,
    "feedback_reward_rerank",
)

comparison_metrics_df = pd.DataFrame([baseline_agg, feedback_agg])

display(comparison_metrics_df)

,method,num_candidates,mrr,precision_at_3,ndcg_at_3,precision_at_5,ndcg_at_5,precision_at_10,ndcg_at_10
0,baseline_rerank,21.626667,0.968167,0.89,0.888482,0.858667,0.896643,0.761333,0.930921
1,feedback_reward_rerank,21.626667,0.969833,0.89,0.891094,0.858000,0.899620,0.764000,0.934588


In [ ]:
comparison_by_query = baseline_metrics_by_query.merge(
    feedback_metrics_by_query,
    on=["query_id", "query"],
    suffixes=("_baseline", "_feedback"),
)

for metric in [
    "mrr",
    "precision_at_3",
    "precision_at_5",
    "precision_at_10",
    "ndcg_at_3",
    "ndcg_at_5",
    "ndcg_at_10",
]:
    b_col = f"{metric}_baseline"
    f_col = f"{metric}_feedback"

    if b_col in comparison_by_query.columns and f_col in comparison_by_query.columns:
        comparison_by_query[f"delta_{metric}"] = comparison_by_query[f_col] - comparison_by_query[b_col]

display(comparison_by_query.head(20))

print("Mean deltas:")
delta_cols = [c for c in comparison_by_query.columns if c.startswith("delta_")]
display(comparison_by_query[delta_cols].mean().to_frame("mean_delta"))

,query_id,query,score_col_baseline,num_candidates_baseline,mrr_baseline,precision_at_3_baseline,ndcg_at_3_baseline,precision_at_5_baseline,ndcg_at_5_baseline,precision_at_10_baseline,ndcg_at_10_baseline,score_col_feedback,num_candidates_feedback,mrr_feedback,precision_at_3_feedback,ndcg_at_3_feedback,precision_at_5_feedback,ndcg_at_5_feedback,precision_at_10_feedback,ndcg_at_10_feedback,delta_mrr,delta_precision_at_3,delta_precision_at_5,delta_precision_at_10,delta_ndcg_at_3,delta_ndcg_at_5,delta_ndcg_at_10
0,0,gluten free options restaurant in Marseille,final_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,0.8,0.996818,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,0.8,0.996818,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
1,1,vegan options restaurant in Vienna,final_rerank_score,24,1.0,1.000000,1.000000,1.0,0.815151,1.0,0.954622,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,0.815151,1.0,0.958492,0.0,0.0,0.0,0.0,0.000000,0.000000,0.003871
2,2,can you suggest a vegetarian friendly italian place in Nice,final_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,1.0,0.949325,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,1.0,0.949325,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
3,3,cheap seafood restaurant in Nice,final_rerank_score,16,0.5,0.666667,0.693426,0.4,0.693426,0.2,0.693426,feedback_rerank_score,16,0.5,0.666667,0.693426,0.4,0.693426,0.2,0.693426,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
4,4,can you suggest a gluten free options fast food place in Prague,final_rerank_score,21,1.0,1.000000,1.000000,1.0,1.000000,0.7,0.966160,feedback_rerank_score,21,1.0,1.000000,1.000000,1.0,1.000000,0.7,0.966160,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
5,5,any good mid french spots in Marseille,final_rerank_score,24,1.0,1.000000,1.000000,1.0,0.912530,1.0,0.943238,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,0.902621,1.0,0.936808,0.0,0.0,0.0,0.0,0.000000,-0.009909,-0.006430
6,6,can you recommend a mid restaurant in Porto,final_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,1.0,0.948924,feedback_rerank_score,24,1.0,1.000000,1.000000,1.0,1.000000,1.0,0.947091,0.0,0.0,0.0,0.0,0.000000,0.000000,-0.001833
7,7,expensive restaurant with family friendly in Granada,final_rerank_score,16,1.0,0.333333,0.613147,0.4,0.877215,0.2,0.877215,feedback_rerank_score,16,1.0,0.333333,0.613147,0.4,0.877215,0.2,0.877215,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
8,8,cheap mediterranean restaurant in Paris,final_rerank_score,24,1.0,1.000000,0.687148,1.0,0.676514,1.0,0.858026,feedback_rerank_score,24,1.0,1.000000,0.687148,1.0,0.676514,1.0,0.858026,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
9,9,i need a expensive place in Vienna with family friendly,final_rerank_score,16,1.0,0.333333,1.000000,0.2,1.000000,0.1,1.000000,feedback_rerank_score,16,1.0,0.333333,1.000000,0.2,1.000000,0.1,1.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000


Mean deltas:


,mean_delta
delta_mrr,0.001667
delta_precision_at_3,0.000000
delta_precision_at_5,-0.000667
delta_precision_at_10,0.002667
delta_ndcg_at_3,0.002613
delta_ndcg_at_5,0.002977
delta_ndcg_at_10,0.003667


In [ ]:
def show_ranking_comparison(query_id, n=10):
    group = feedback_labeled_df[feedback_labeled_df["query_id"] == query_id].copy()

    if len(group) == 0:
        print("Query id not found:", query_id)
        return

    baseline = group.sort_values("final_rerank_score", ascending=False).head(n).copy()
    feedback_ranked = group.sort_values("feedback_rerank_score", ascending=False).head(n).copy()

    cols = [
        "name",
        "city_filled",
        "country",
        "price_bucket",
        "rating",
        "feedback",
        "relevance",
        "hidden_utility",
        "semantic_score_norm",
        "metadata_score_norm",
        "rating_score",
        "popularity_score_norm",
        "soft_constraint_score",
        "hard_constraint_score",
        "final_rerank_score",
        "reward_score",
        "feedback_rerank_score",
    ]

    cols = [c for c in cols if c in group.columns]

    print("=" * 120)
    print("QUERY_ID:", query_id)
    print("QUERY:", group["query"].iloc[0])
    print("PARSED:", group["parsed_query_json"].iloc[0])

    print("\nBaseline ranking:")
    display(baseline[cols])

    print("\nFeedback reward ranking:")
    display(feedback_ranked[cols])


for qid in feedback_labeled_df["query_id"].drop_duplicates().head(3):
    show_ranking_comparison(qid, n=10)

QUERY_ID: 0
QUERY: gluten free options restaurant in Marseille
PARSED: {"city": "marseille", "country": null, "price_bucket": null, "tags": [], "dietary": ["gluten free options"], "matched_meals": [], "matched_features": []}

Baseline ranking:


,name,city_filled,country,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,reward_score,feedback_rerank_score
5,Bistrot o'prado,Marseille,France,mid,5.0,1,2,1.000000,0.943690,1.0,1.0,0.664194,1.0,1.0,0.943895,0.987105,0.956858
6,Noodles Chôp,Marseille,France,mid,4.5,1,2,0.896314,0.935780,1.0,0.9,0.806574,1.0,1.0,0.939969,0.984644,0.953372
7,Cafe Bovo,Marseille,France,expensive,5.0,1,2,0.883945,0.899427,1.0,1.0,0.745240,1.0,1.0,0.934295,0.988205,0.950468
0,Bien Etre et Petit Plat,Marseille,France,expensive,4.5,1,2,0.894502,0.977362,1.0,0.9,0.189023,1.0,1.0,0.894847,0.960283,0.914478
4,Green Love,Marseille,France,expensive,4.5,1,2,1.000000,0.910167,1.0,0.9,0.389458,1.0,1.0,0.888012,0.969226,0.912377
1,La Broceliande,Marseille,France,mid,4.0,1,2,0.848353,0.852595,1.0,0.8,0.224041,1.0,1.0,0.833442,0.938094,0.864838
2,Cafe Borely,Marseille,France,mid,3.5,0,1,0.651879,0.893031,1.0,0.7,0.149436,1.0,1.0,0.827156,0.903012,0.849913
3,waoreng Bali,Marseille,France,mid,4.0,1,2,0.863221,0.865859,1.0,0.8,0.020287,1.0,1.0,0.818372,0.916521,0.847817
11,La Boite a Panisse,Marseille,France,cheap,4.5,-1,0,0.396024,0.263071,0.0,0.9,0.304390,0.0,1.0,0.270667,0.021536,0.195928
15,AU Doyen,Marseille,France,mid,4.0,-1,0,0.398121,0.243918,0.0,0.8,0.350260,0.0,1.0,0.252593,0.015322,0.181412



Feedback reward ranking:


,name,city_filled,country,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,reward_score,feedback_rerank_score
5,Bistrot o'prado,Marseille,France,mid,5.0,1,2,1.000000,0.943690,1.0,1.0,0.664194,1.0,1.0,0.943895,0.987105,0.956858
6,Noodles Chôp,Marseille,France,mid,4.5,1,2,0.896314,0.935780,1.0,0.9,0.806574,1.0,1.0,0.939969,0.984644,0.953372
7,Cafe Bovo,Marseille,France,expensive,5.0,1,2,0.883945,0.899427,1.0,1.0,0.745240,1.0,1.0,0.934295,0.988205,0.950468
0,Bien Etre et Petit Plat,Marseille,France,expensive,4.5,1,2,0.894502,0.977362,1.0,0.9,0.189023,1.0,1.0,0.894847,0.960283,0.914478
4,Green Love,Marseille,France,expensive,4.5,1,2,1.000000,0.910167,1.0,0.9,0.389458,1.0,1.0,0.888012,0.969226,0.912377
1,La Broceliande,Marseille,France,mid,4.0,1,2,0.848353,0.852595,1.0,0.8,0.224041,1.0,1.0,0.833442,0.938094,0.864838
2,Cafe Borely,Marseille,France,mid,3.5,0,1,0.651879,0.893031,1.0,0.7,0.149436,1.0,1.0,0.827156,0.903012,0.849913
3,waoreng Bali,Marseille,France,mid,4.0,1,2,0.863221,0.865859,1.0,0.8,0.020287,1.0,1.0,0.818372,0.916521,0.847817
11,La Boite a Panisse,Marseille,France,cheap,4.5,-1,0,0.396024,0.263071,0.0,0.9,0.304390,0.0,1.0,0.270667,0.021536,0.195928
13,La Cuisine Du Dôme,Marseille,France,expensive,5.0,-1,0,0.435095,0.184710,0.0,1.0,0.238809,0.0,1.0,0.247765,0.026686,0.181441


QUERY_ID: 1
QUERY: vegan options restaurant in Vienna
PARSED: {"city": "vienna", "country": null, "price_bucket": null, "tags": [], "dietary": ["vegan options"], "matched_meals": [], "matched_features": []}

Baseline ranking:


...

,name,city_filled,country,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,reward_score,feedback_rerank_score
30,Restaurant Rote Bar,Vienna,Austria,expensive,4.5,1,2,0.992275,0.928839,1.0,0.9,0.551721,1.0,1.0,0.911708,0.976680,0.931199
28,Wiener,Vienna,Austria,expensive,4.5,1,2,0.782308,0.916229,1.0,0.9,0.545792,1.0,1.0,0.906071,0.976174,0.927102
29,Pizzeria Scarabocchio,Vienna,Austria,mid,4.5,1,2,0.948878,0.954220,1.0,0.9,0.394383,1.0,1.0,0.906126,0.970708,0.925501
26,Conditorei Sluka Kärntner Strasse,Vienna,Austria,expensive,4.0,0,1,0.698556,0.966066,1.0,0.8,0.438435,1.0,1.0,0.900270,0.960114,0.918223
25,Formosa,Vienna,Austria,mid,4.0,0,1,0.718880,0.950268,1.0,0.8,0.099850,1.0,1.0,0.860092,0.931427,0.881493
24,Das Salettl,Vienna,Austria,mid,4.0,1,2,0.785035,0.884263,1.0,0.8,0.124992,1.0,1.0,0.836204,0.929960,0.864331
27,Hamama,Vienna,Austria,mid,3.5,0,1,0.731283,0.936586,1.0,0.7,0.093668,1.0,1.0,0.839001,0.898596,0.856880
39,Akakiko,Vienna,Austria,expensive,3.5,1,2,0.852876,0.867623,1.0,0.7,0.134809,1.0,1.0,0.815530,0.898659,0.840469
37,DOTS im Brunnerhof,Vienna,Austria,expensive,3.5,1,2,0.913056,0.862172,1.0,0.7,0.104586,1.0,1.0,0.810327,0.893539,0.835291
31,Health Kitchen,Vienna,Austria,expensive,3.5,1,2,0.910139,0.860801,1.0,0.7,0.072792,1.0,1.0,0.806599,0.888329,0.831118


QUERY_ID: 2
QUERY: can you suggest a vegetarian friendly italian place in Nice
PARSED: {"city": "nice", "country": null, "price_bucket": null, "tags": ["italian"], "dietary": ["vegetarian friendly"], "matched_meals": [], "matched_features": []}

Baseline ranking:


,name,city_filled,country,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,reward_score,feedback_rerank_score
52,La Cucina,Nice,France,expensive,5.0,1,2,0.950817,0.975095,1.0,1.0,0.703613,1.0,1.0,0.960399,0.988263,0.968758
51,Le Rossini,Nice,France,expensive,4.0,1,2,0.978541,0.932556,1.0,0.8,0.298024,1.0,1.0,0.872825,0.948708,0.895589
53,La Focaccia,Nice,France,mid,4.5,1,2,0.963926,0.877778,1.0,0.9,0.292933,1.0,1.0,0.865404,0.963018,0.894688
49,Il Vicoletto,Nice,France,mid,4.0,1,2,0.978784,0.852203,1.0,0.8,0.285302,1.0,1.0,0.839411,0.943673,0.870690
54,Piccola Italia,Nice,France,expensive,3.5,1,2,0.780089,0.894023,1.0,0.7,0.130527,1.0,1.0,0.825662,0.900340,0.848065
48,O'Staff,Nice,France,expensive,3.0,1,2,0.985305,0.909551,1.0,0.6,0.045192,1.0,1.0,0.808339,0.841854,0.818394
55,Bocca Buona,Nice,France,expensive,3.5,1,2,0.916117,0.867339,1.0,0.7,0.057447,1.0,1.0,0.807680,0.886442,0.831309
50,Cafe de La Place,Nice,France,expensive,3.0,0,1,0.711175,0.857413,1.0,0.6,0.043160,1.0,1.0,0.787281,0.834535,0.801458
61,Les délices d'Aladin,Nice,France,mid,4.5,0,1,0.631263,0.555656,0.5,0.9,0.172657,0.5,1.0,0.549528,0.377566,0.497939
63,Pizza Com d'hab,Nice,France,expensive,4.5,0,1,0.649614,0.570782,0.5,0.9,0.060598,0.5,1.0,0.544373,0.338607,0.482643



Feedback reward ranking:


,name,city_filled,country,price_bucket,rating,feedback,relevance,hidden_utility,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,reward_score,feedback_rerank_score
52,La Cucina,Nice,France,expensive,5.0,1,2,0.950817,0.975095,1.0,1.0,0.703613,1.0,1.0,0.960399,0.988263,0.968758
51,Le Rossini,Nice,France,expensive,4.0,1,2,0.978541,0.932556,1.0,0.8,0.298024,1.0,1.0,0.872825,0.948708,0.895589
53,La Focaccia,Nice,France,mid,4.5,1,2,0.963926,0.877778,1.0,0.9,0.292933,1.0,1.0,0.865404,0.963018,0.894688
49,Il Vicoletto,Nice,France,mid,4.0,1,2,0.978784,0.852203,1.0,0.8,0.285302,1.0,1.0,0.839411,0.943673,0.870690
54,Piccola Italia,Nice,France,expensive,3.5,1,2,0.780089,0.894023,1.0,0.7,0.130527,1.0,1.0,0.825662,0.900340,0.848065
55,Bocca Buona,Nice,France,expensive,3.5,1,2,0.916117,0.867339,1.0,0.7,0.057447,1.0,1.0,0.807680,0.886442,0.831309
48,O'Staff,Nice,France,expensive,3.0,1,2,0.985305,0.909551,1.0,0.6,0.045192,1.0,1.0,0.808339,0.841854,0.818394
50,Cafe de La Place,Nice,France,expensive,3.0,0,1,0.711175,0.857413,1.0,0.6,0.043160,1.0,1.0,0.787281,0.834535,0.801458
61,Les délices d'Aladin,Nice,France,mid,4.5,0,1,0.631263,0.555656,0.5,0.9,0.172657,0.5,1.0,0.549528,0.377566,0.497939
63,Pizza Com d'hab,Nice,France,expensive,4.5,0,1,0.649614,0.570782,0.5,0.9,0.060598,0.5,1.0,0.544373,0.338607,0.482643


In [ ]:
def apply_feedback_reward_reranking(
    reranked_df,
    reward_model,
    feature_cols=FEATURE_COLS,
    top_k=5,
):
    df_out = reranked_df.copy()

    for col in feature_cols:
        if col not in df_out.columns:
            df_out[col] = 0.0

        df_out[col] = pd.to_numeric(
            df_out[col],
            errors="coerce",
        ).fillna(0.0)

    if "final_rerank_score" not in df_out.columns:
        df_out["final_rerank_score"] = 0.0

    df_out["final_rerank_score"] = pd.to_numeric(
        df_out["final_rerank_score"],
        errors="coerce",
    ).fillna(0.0)

    df_out["reward_score"] = reward_model.predict_proba(
        df_out[feature_cols].values
    )[:, 1]

    df_out["feedback_rerank_score"] = (
        FEEDBACK_RERANK_WEIGHTS["original_rerank"] * df_out["final_rerank_score"].astype(float)
        + FEEDBACK_RERANK_WEIGHTS["reward_score"] * df_out["reward_score"].astype(float)
    )

    df_out = df_out.sort_values(
        ["feedback_rerank_score", "reward_score", "final_rerank_score"],
        ascending=False,
    ).reset_index(drop=True)

    df_out.insert(0, "feedback_rank_position", np.arange(1, len(df_out) + 1))

    if top_k is not None:
        df_out = df_out.head(top_k).copy().reset_index(drop=True)

    return df_out


sample_qid = feedback_labeled_df["query_id"].iloc[0]
sample_group = feedback_labeled_df[feedback_labeled_df["query_id"] == sample_qid].copy()

demo_feedback_ranked = apply_feedback_reward_reranking(
    sample_group,
    reward_model,
    top_k=5,
)

display(demo_feedback_ranked[
    [
        "feedback_rank_position",
        "query",
        "name",
        "final_rerank_score",
        "reward_score",
        "feedback_rerank_score",
        "relevance",
        "feedback",
    ]
])

,feedback_rank_position,query,name,final_rerank_score,reward_score,feedback_rerank_score,relevance,feedback
0,1,gluten free options restaurant in Marseille,Bistrot o'prado,0.943895,0.987105,0.956858,2,1
1,2,gluten free options restaurant in Marseille,Noodles Chôp,0.939969,0.984644,0.953372,2,1
2,3,gluten free options restaurant in Marseille,Cafe Bovo,0.934295,0.988205,0.950468,2,1
3,4,gluten free options restaurant in Marseille,Bien Etre et Petit Plat,0.894847,0.960283,0.914478,2,1
4,5,gluten free options restaurant in Marseille,Green Love,0.888012,0.969226,0.912377,2,1


In [ ]:
REWARD_MODEL_PATH = MODEL_DIR / "feedback_reward_model.joblib"
REWARD_FEATURES_PATH = MODEL_DIR / "reward_feature_columns.json"

joblib.dump(reward_model, REWARD_MODEL_PATH)

with open(REWARD_FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "feature_cols": FEATURE_COLS,
            "feedback_rerank_weights": FEEDBACK_RERANK_WEIGHTS,
            "feedback_source": feedback_source,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved reward model:", REWARD_MODEL_PATH)
print("Saved feature config:", REWARD_FEATURES_PATH)

Saved reward model: /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/reward_model/feedback_reward_model.joblib
Saved feature config: /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/reward_model/reward_feature_columns.json


In [ ]:
FEEDBACK_DATASET_PATH = DATA_DIR / "synthetic_feedback_labeled_dataset.csv"
COMPARISON_METRICS_PATH = METRICS_DIR / "feedback_reranking_comparison_metrics.csv"
BASELINE_BY_QUERY_PATH = METRICS_DIR / "baseline_metrics_by_query.csv"
FEEDBACK_BY_QUERY_PATH = METRICS_DIR / "feedback_metrics_by_query.csv"
COMPARISON_BY_QUERY_PATH = METRICS_DIR / "comparison_by_query.csv"
COEF_PATH = METRICS_DIR / "reward_model_coefficients.csv"

feedback_labeled_df.to_csv(FEEDBACK_DATASET_PATH, index=False)
comparison_metrics_df.to_csv(COMPARISON_METRICS_PATH, index=False)
baseline_metrics_by_query.to_csv(BASELINE_BY_QUERY_PATH, index=False)
feedback_metrics_by_query.to_csv(FEEDBACK_BY_QUERY_PATH, index=False)
comparison_by_query.to_csv(COMPARISON_BY_QUERY_PATH, index=False)
coef_df.to_csv(COEF_PATH, index=False)

print("Saved:")
print("-", FEEDBACK_DATASET_PATH)
print("-", COMPARISON_METRICS_PATH)
print("-", BASELINE_BY_QUERY_PATH)
print("-", FEEDBACK_BY_QUERY_PATH)
print("-", COMPARISON_BY_QUERY_PATH)
print("-", COEF_PATH)

Saved:
- /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/data/synthetic_feedback_labeled_dataset.csv
- /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/metrics/feedback_reranking_comparison_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/metrics/baseline_metrics_by_query.csv
- /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/metrics/feedback_metrics_by_query.csv
- /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/metrics/comparison_by_query.csv
- /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/metrics/reward_model_coefficients.csv


In [ ]:
feedback_rlhf_config = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "method": "RLHF-like preference reward model for recommendation reranking",
    "note": (
        "This is not PPO-based RLHF on the language model. "
        "It is a feedback-driven reward model trained on synthetic preference labels "
        "and used to improve recommendation reranking."
    ),
    "feedback_source": feedback_source,
    "num_feedback_rows": int(len(feedback_labeled_df)),
    "num_queries": int(feedback_labeled_df["query_id"].nunique()),
    "reward_model_type": "LogisticRegression over non-circular reranking features",
    "feature_cols": FEATURE_COLS,
    "feedback_rerank_weights": FEEDBACK_RERANK_WEIGHTS,
    "reward_model_path": str(REWARD_MODEL_PATH),
    "comparison_metrics_path": str(COMPARISON_METRICS_PATH),
    "synthetic_feedback_generation": {
        "num_synthetic_queries": N_SYNTHETIC_QUERIES,
        "positive_candidates_per_query": N_POSITIVE_PER_QUERY,
        "partial_candidates_per_query": N_PARTIAL_PER_QUERY,
        "negative_candidates_per_query": N_NEGATIVE_PER_QUERY,
        "hidden_utility": (
            "0.30 hard_constraint + 0.25 soft_constraint + 0.15 metadata "
            "+ 0.20 rating + 0.10 popularity + gaussian noise"
        ),
    },
}

FEEDBACK_RLHF_CONFIG_PATH = OUTPUT_DIR / "feedback_rlhf_config.json"

with open(FEEDBACK_RLHF_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(feedback_rlhf_config, f, indent=2, ensure_ascii=False)

print("Saved config:", FEEDBACK_RLHF_CONFIG_PATH)
feedback_rlhf_config

Saved config: /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/feedback_rlhf_config.json


{'created_at': '2026-05-06T19:23:55.830374Z',
 'method': 'RLHF-like preference reward model for recommendation reranking',
 'note': 'This is not PPO-based RLHF on the language model. It is a feedback-driven reward model trained on synthetic preference labels and used to improve recommendation reranking.',
 'feedback_source': 'synthetic_large_non_circular_fast',
 'num_feedback_rows': 6488,
 'num_queries': 300,
 'reward_model_type': 'LogisticRegression over non-circular reranking features',
 'feature_cols': ['semantic_score_norm',
  'metadata_score_norm',
  'rating_score',
  'popularity_score_norm',
  'soft_constraint_score',
  'hard_constraint_score'],
 'feedback_rerank_weights': {'original_rerank': 0.7, 'reward_score': 0.3},
 'reward_model_path': '/content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/reward_model/feedback_reward_model.joblib',
 'comparison_metrics_path': '/content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/metrics/feedback_reranking_comparison_metrics.c